In [ ]:
import logging
from tensorflow.python.keras.preprocessing import sequence
from tensorflow.python.keras import Sequential
import keras.layers
import tensorflow as tf
import random
import keras
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
import numpy as np
from numpy import newaxis
import pandas as pd
import tables
import os
import scipy.io as spio
from sklearn.model_selection import KFold

def getLogger():
    logHandler = logging.StreamHandler()
    # Formats logger to output time, function being called, and line number (for debugging)
    FORMAT = '%(asctime)-15s %(levelname)s %(funcName)s [%(lineno)d] %(message)s'
    logFormatter = logging.Formatter(FORMAT)
    logHandler.setFormatter(logFormatter)
    logger = logging.getLogger()
    # Convert to info to debug to debug code
    logger.setLevel(os.environ.get("LOGLEVEL", "INFO"))
    logger.addHandler(logHandler)
    return logger

def getMaxDimensions(strpath):
    neuronPadVal = []
    for root, dirs, files in os.walk(strpath):
        for file in files:
            if file.endswith('.mat'):
                checkfile = os.path.join(root, file)
                checkSet = loadvalues(checkfile)
                neuronPadVal.append((np.shape(checkSet[0]))[1])
    maxNeuronNum = max(neuronPadVal)
    return maxNeuronNum

def processForPadding(trainingSet, maxNeuronNum):
    # Determines the number of times neurons must be padded in
    padNumber = abs(maxNeuronNum - np.shape(trainingSet[0])[1])
    # Pads along the neuron axis adding 200 at every index
    trainingSet[0] = np.pad(trainingSet[0], ((0,0),(0,padNumber),(0,0)), 'constant', constant_values=100000)
    # Used to ensure all input dimensions are the same
    return trainingSet

def duplicateSet(inpSet,numForDup):
    trainArt = inpSet[0]
    testArt = inpSet[1]
    for j in range(numForDup):
        trainArt = np.append(trainArt, inpSet[0], axis = 0)
        testArt = np.append(testArt, inpSet[1], axis = 0)
    return [trainArt, testArt]

#Modify for new variable names
def loadvalues(filepath):
    mat = spio.loadmat(filepath)
    # Sets all values in sessMat to a float type, NaN is considered none
    features = mat.get('features')
    features = features.astype(float)
    # Sets all values in sessTrials to an int type
    behavior = mat.get('NorthSouth')
    behavior = behavior.flatten()
    behavior = behavior.astype(int)
    behaviorVectorized = []
    for i in range(len(behavior)):
        if behavior[i] == 0:
            behaviorVectorized.append([0,1])
        elif behavior[i] == 1:
            behaviorVectorized.append([1,0])
    return [features,behaviorVectorized]

def buildmodel():
    # Creates model with 2 LSTM layers and one Dense layer with sigmoid activation compressing to two values
    # representative of behavior
    # model.add(LSTM(np.shape(a_train)[1], dropout=0.2, recurrent_dropout=0.2))
    # model.add(Dense(1, activation='sigmoid'))
    model = keras.models.Sequential([
        keras.layers.Masking(mask_value=100000),
        keras.layers.LSTM(
            60, #Used 60 in No_Norm,Z_NoInter
            return_sequences=True, 
            dropout=0.2, # Vary as needed <0.2
            recurrent_dropout=0.2, # Vary as needed < 0.2
        ),
        keras.layers.LSTM(
            30, #Used 30 in No_Norm,Z_NoInter 
            return_sequences=True, 
            dropout=0.2, # Vary as needed <0.2
            recurrent_dropout=0.2, # Vary as needed <0.2
        ),
        keras.layers.LSTM(2,activation='softmax'),
    ])
    # Model is compiled with adam optimizer, can be changed per discretion
    # Loss is set to binary crossentropy as final behavior is either 0 or 1 based on our operationalization
    # Optimizer documentation at https://keras.io/optimizers/
    opt = keras.optimizers.RMSprop(
        lr=0.001, 
        rho=0.9, 
        epsilon=None, 
        decay=0.0,
    )
    model.compile(
        loss='categorical_crossentropy',
        optimizer=opt,
        metrics=['accuracy']
    )
    return model

def modelTrainer(filesPath,validationFilesPath,fileModelPath):
    maxNeuronNum = getMaxDimensions(filesPath)
    for modelRoot, modelDirs, modelFiles in os.walk(filesPath):
        for modelFile in modelFiles:
            print(f"--------------------------------\nWorking on {modelFile}\n")
            modelFileName = modelFile[:-13]
            # Gets training files in directory of interest
            if modelFile.endswith('.mat'):
                trainingFile = os.path.join(modelRoot, modelFile)
                trainingSet = loadvalues(trainingFile)
                trainingSet = processForPadding(trainingSet, maxNeuronNum)
                trainingSetShape = np.shape(trainingSet[0])
                #Ensures data is actually present in the training files
                #Some may have no correct trials hence compresses to 0 based on our filtering
                if trainingSetShape[1] == 0:
                    print(f"': Unable to train, zero measurements'\n")
                else:
                    #Sets up event log to later visualize files in TensorBoard
                    #logdir = fileTensorPath
                    #tensorboard_callback = keras.callbacks.TensorBoard(log_dir = logdir)
                    trainingSet = duplicateSet(trainingSet,50)
                    modelSavename = os.path.join(fileModelPath,f'{modelFileName}.h5')
                    if 'Saline' in filesPath:
                        for valRoot, valDirs, valFiles in os.walk(validationFilesPath):
                            for valFile in valFiles:
                                if modelFileName in valFile:
                                    if 'East' in valFile:
                                        tmpTestingFileE = os.path.join(valRoot,valFile)
                                        tmpTestingSetE = loadvalues(tmpTestingFileE)
                                        tmpTestingSetE = processForPadding(tmpTestingSetE,maxNeuronNum)
                        valDataXEast = tmpTestingSetE[0]
                        valDataYEast = tmpTestingSetE[1]
                        valDataYEast = np.reshape(valDataYEast,(np.shape(tmpTestingSetE[1])[0],np.shape(tmpTestingSetE[1])[1]))
                        for valRoot, valDirs, valFiles in os.walk(validationFilesPath):
                            for valFile in valFiles:
                                if modelFileName in valFile:
                                    if 'West' in valFile:
                                        tmpTestingFileW = os.path.join(valRoot,valFile)
                                        tmpTestingSetW = loadvalues(tmpTestingFileW)
                                        tmpTestingSetW = processForPadding(tmpTestingSetW,maxNeuronNum)
                        valDataXWest = tmpTestingSetW[0]
                        valDataYWest = tmpTestingSetW[1]
                        valDataYWest = np.reshape(valDataYWest,(np.shape(tmpTestingSetW[1])[0],np.shape(tmpTestingSetW[1])[1]))
                        valDataX = np.concatenate((valDataXEast,valDataXWest),axis=0)
                        valDataY = np.concatenate((valDataYEast,valDataYWest),axis=0)
                    # Defines and initializes model
                    model = buildmodel()
                    es = EarlyStopping(monitor='loss', mode='min', verbose=2, patience=10)
                    if 'Muscimol' in filesPath:
                        mc = ModelCheckpoint(modelSavename, monitor='loss', mode='min', verbose=2, save_best_only=True)
                        model.fit(trainingSet[0],
                                  trainingSet[1],
                                  batch_size=int(np.shape(trainingSet[0])[0]/75),
                                  epochs=75,
                                  verbose=2,
                                  callbacks=[es,mc],
                                 )
                              #callbacks=[tensorboard_callback]
                    elif 'Saline' in filesPath:
                        mc = ModelCheckpoint(modelSavename, monitor='val_loss', mode='min', verbose=2, save_best_only=True)
                        model.fit(trainingSet[0],
                                  trainingSet[1],
                                  validation_data=(valDataX,valDataY),
                                  batch_size=int(np.shape(trainingSet[0])[0]/75),
                                  epochs=75,
                                  verbose=2,
                                  callbacks=[es,mc],
                                 )
                              #callbacks=[tensorboard_callback]
                    print(f"--------------------------------\nModel saved from {modelFile}\n")
    return 0

filesPath=['',
          ]

fileModelPath = ['',
                ]

trainingFiles = []

modelNames = [
             ]

dupeSetNums = [
               [],
              ]
 
for j in range(np.shape(trainingFiles)[0]):
    for i in range(len(trainingFiles[j])):
        maxNeuronNum = getMaxDimensions(filesPath[j])
        trainingSet = loadvalues((trainingFiles[j])[i])
        trainingSet = processForPadding(trainingSet, maxNeuronNum)
        trainingSetShape = np.shape(trainingSet[0])
        if trainingSetShape[1] == 0:
            print(f"': Unable to train, zero measurements'\n")
        else:
            #Sets up event log to later visualize files in TensorBoard
            #logdir = fileTensorPath
            #tensorboard_callback = keras.callbacks.TensorBoard(log_dir = logdir)
            trainingSet = duplicateSet(trainingSet,(dupeSetNums[j])[i])
            modelSavename = os.path.join(fileModelPath[j],(modelNames[j])[i])

            # Defines and initializes model
            model = buildmodel()
            es = EarlyStopping(monitor='loss', mode='min', verbose=2, patience=10)
            mc = ModelCheckpoint(modelSavename, monitor='val_loss', mode='min', verbose=2, save_best_only=True)
            model.fit(trainingSet[0],
                      trainingSet[1],
                      validation_split=0.15,
                      batch_size=int(np.shape(trainingSet[0])[0]/75),
                      epochs=75,
                      verbose=2,
                      callbacks=[es,mc],
                     )
            del model
